# Task 1: Rotated MNIST

In this task you will use MNIST dataset. The images are 28x28 and they are **rotated** by an angle from the range (-100, 100).  
You are given a pipeline that trains a multi-head convolutional neural network on this dataset. The first head of the model performs a digit classification and the second head tries to predict the angle that the digit was rotated by.

Your task is to:
1. **(5 pts)** Implement CNN with classification and regression heads.
2. **(3 pts)** Implement the model's loss - select appropriate loss functions for both heads and combine them into final loss of the model.
3. **(3 pts)** Check the model's predictions on the test data and find for which classes the model achieves the best/worst performance both for classification and regression. Then, write a short explanation for the observed model behavior (why does the model have a problem with particular classes?).

Hints:
- You don't need to create a very sophisticated model - a few convolutions for CNN and a few linear layers for heads should be enough. After a few epochs the model should achieve classification accuracy above 90% and regression MAE below 18 on a test dataset.
- When training multi-head models it is usually good to scale losses from particular heads so that they have similar contribution towards the final loss.

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from PIL import Image

In [98]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        # CNN backbone with pooling to reduce dimensions
        self.conv_blocks = torch.nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1),   # 28x28 -> 28x28
            nn.ReLU(),
            nn.MaxPool2d(2),                   # 28x28 -> 14x14
            nn.Conv2d(32, 64, 3, padding=1),  # 14x14 -> 14x14
            nn.ReLU(),
            nn.MaxPool2d(2),                   # 14x14 -> 7x7
            nn.Conv2d(64, 128, 3, padding=1), # 7x7 -> 7x7
            nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),           # 7x7 -> 1x1 (128 features)
            nn.Flatten(),
        )
        
        # Classification head (10 digit classes)
        self.classification_head = torch.nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 10),
        )
        
        # Regression head (predict rotation angle)
        self.regression_head = torch.nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        # Shared CNN features
        features = self.conv_blocks(x)
        
        # Two separate heads
        log_probs = F.log_softmax(self.classification_head(features), dim=1)
        angle = self.regression_head(features).squeeze(1)  # Remove last dimension
        
        return log_probs, angle


def train(model, device, train_loader, optimizer, epoch, log_interval):
    model.train()
    for batch_idx, (data, (target_digit, target_angle)) in enumerate(train_loader):
        data = data.to(device)
        target_digit, target_angle = target_digit.to(device), target_angle.to(device)
        optimizer.zero_grad()
        log_probs, angle = model(data)
        
        # Classification loss: NLL loss (works with log_softmax output)
        classification_loss = F.nll_loss(log_probs, target_digit)
        
        # Regression loss: MSE or L1 (MAE) - L1 is more robust to outliers
        regression_loss = F.mse_loss(angle, target_angle)
        
        # Scale regression loss to balance with classification loss
        # Angles are in range [-100, 100], MSE can be very large
        # Scaling factor makes both losses contribute similarly
        loss = classification_loss + 0.005 * regression_loss
        
        loss.backward()
        optimizer.step()
        
        if (batch_idx + 1) % log_interval == 0:
            print('Train Epoch: {} [{}/{} ({:.0f}%)]\tLoss: {:.6f} (cls: {:.4f}, reg: {:.4f})'.format(
                epoch, (batch_idx + 1) * len(data), len(train_loader.dataset),
                100. * (batch_idx + 1) / len(train_loader), loss.item(),
                classification_loss.item(), regression_loss.item()
            ))


def test(model, device, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    abs_error = 0
    with torch.no_grad():
        for data, (target_digit, target_angle) in test_loader:
            data = data.to(device)
            target_digit, target_angle = target_digit.to(device), target_angle.to(device)
            log_probs, angle = model(data)
            
            # Same loss as training
            classification_loss = F.nll_loss(log_probs, target_digit, reduction='sum')
            regression_loss = F.mse_loss(angle, target_angle, reduction='sum')
            test_loss += classification_loss + 0.005 * regression_loss
        
            pred_digit = log_probs.argmax(dim=1, keepdim=True)
            correct += pred_digit.eq(target_digit.view_as(pred_digit)).sum().item()
            abs_error += (target_angle - angle).abs().sum().item()

    test_loss /= len(test_loader.dataset)
    print('\nTest set: Average loss: {:.4f}, Classification accuracy: {}/{} ({:.0f}%), Regression MAE: {:.2f}\n'.format(
        test_loss, correct, len(test_loader.dataset),
        100. * correct / len(test_loader.dataset),
        abs_error / len(test_loader.dataset)
    ))

In [93]:
batch_size = 256
test_batch_size = 1000
epochs = 5
lr = 3e-3
seed = 1
log_interval = 50
use_cuda = torch.cuda.is_available()

In [94]:
torch.manual_seed(seed)
device = torch.device("cuda" if use_cuda else "cpu")

train_kwargs = {'batch_size': batch_size}
test_kwargs = {'batch_size': test_batch_size}
if use_cuda:
    cuda_kwargs = {
        'num_workers': 1,
        'pin_memory': True,
        'shuffle': True
    }
    train_kwargs.update(cuda_kwargs)
    test_kwargs.update(cuda_kwargs)

In [95]:
class MNISTWithRotations(datasets.MNIST):
    def __init__(self, *args, transform=None, target_transform=None, **kwargs):
        super(MNISTWithRotations, self).__init__(*args, **kwargs)
        self.rotation_angles = (torch.rand(len(self.data)) - 0.5) * 2 * 100
        self.is_img_transforemed = [False] * len(self.data)
        self.transformed_data = torch.zeros(*self.data.shape)

    def __getitem__(self, idx):
        if not self.is_img_transforemed[idx]:
            transform = transforms.Compose([
                transforms.ToTensor(),
                transforms.Lambda(
                    lambda x: transforms.functional.rotate(x, self.rotation_angles[idx].item())
                ),
                transforms.Normalize((0.1307,), (0.3081,)),
            ])
            img = Image.fromarray(self.data[idx].numpy(), mode="L")
            self.transformed_data[idx] = transform(img)
            self.is_img_transforemed[idx] = True

        img = self.transformed_data[idx].unsqueeze(0)
        target_digit = int(self.targets[idx])
        target_angle = self.rotation_angles[idx]
        return img, (target_digit, target_angle)

In [96]:
train_dataset = MNISTWithRotations('../data', train=True, download=True)
test_dataset = MNISTWithRotations('../data', train=False)

train_loader = torch.utils.data.DataLoader(train_dataset, **train_kwargs)
test_loader = torch.utils.data.DataLoader(test_dataset, **test_kwargs)

In [99]:
model = Net().to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
for epoch in range(1, epochs + 1):
    train(model, device, train_loader, optimizer, epoch, log_interval)
    test(model, device, test_loader)

Train Epoch: 1 [12800/60000 (21%)]	Loss: 16.021978 (cls: 2.0069, reg: 2803.0171)
Train Epoch: 1 [25600/60000 (43%)]	Loss: 13.461702 (cls: 1.9020, reg: 2311.9321)
Train Epoch: 1 [38400/60000 (64%)]	Loss: 8.629195 (cls: 1.8172, reg: 1362.3997)
Train Epoch: 1 [51200/60000 (85%)]	Loss: 8.180447 (cls: 1.8322, reg: 1269.6431)

Test set: Average loss: 9.6857, Classification accuracy: 3398/10000 (34%), Regression MAE: 31.30

Train Epoch: 2 [12800/60000 (21%)]	Loss: 10.102373 (cls: 1.8908, reg: 1642.3240)
Train Epoch: 2 [25600/60000 (43%)]	Loss: 6.436417 (cls: 1.7902, reg: 929.2446)
Train Epoch: 2 [38400/60000 (64%)]	Loss: 6.539292 (cls: 1.8627, reg: 935.3178)
Train Epoch: 2 [51200/60000 (85%)]	Loss: 7.439788 (cls: 1.7973, reg: 1128.4990)

Test set: Average loss: 5.4804, Classification accuracy: 3641/10000 (36%), Regression MAE: 21.04

Train Epoch: 3 [12800/60000 (21%)]	Loss: 4.543719 (cls: 1.6751, reg: 573.7313)
Train Epoch: 3 [25600/60000 (43%)]	Loss: 5.123697 (cls: 1.7037, reg: 683.9957)
Tra

# Analysis of model performance for particular classes

In [ ]:
############## TODO ################
# Subtask 3: Check the model's predictions on the test data and find for which classes the
# model achieves the best/worst performance both for classification and regression.
# Write short explanation for observed behavior

In [ ]:
# Analyze per-class performance
model.eval()
class_correct = [0] * 10
class_total = [0] * 10
class_angle_error = [0.0] * 10

with torch.no_grad():
    for data, (target_digit, target_angle) in test_loader:
        data = data.to(device)
        target_digit, target_angle = target_digit.to(device), target_angle.to(device)
        log_probs, angle = model(data)
        
        pred_digit = log_probs.argmax(dim=1)
        
        for i in range(len(target_digit)):
            label = target_digit[i].item()
            class_total[label] += 1
            class_correct[label] += (pred_digit[i] == label).item()
            class_angle_error[label] += abs(angle[i].item() - target_angle[i].item())

# Print results
print("Per-class Classification Accuracy:")
for i in range(10):
    acc = 100 * class_correct[i] / class_total[i]
    mae = class_angle_error[i] / class_total[i]
    print(f"Digit {i}: {acc:.1f}% accuracy, {mae:.2f} MAE")